# AI-Chip Market Intelligence
## Introduction
**Multivariate Forecasting, Dynamic Interdependencies and Volatility Risk Analysis
of NVIDIA, AMD and the Semiconductor Market.**

Mini-projet TEK-UP, Séries Temporelles pour Data Science & IA, 2026/2027.
Comment prévoir le prix ajusté NVDA à **5 et 10 séances**, comprendre les liens
dynamiques avec **AMD et SOXX**, et décrire le risque de volatilité ?
Prix, interdépendances et variance sont trois questions statistiques différentes.

Ce notebook raconte les Phases 1–8 et vérifie leurs résultats publiés.
Toutes les valeurs affichées proviennent des fichiers cités, protégés par SHA256.
Les calculs ci-dessous sont des contrôles arithmétiques des sorties gelées,
sans nouvelle prévision, sélection ou réévaluation du holdout.

**Mode A — REVIEW / FAST** : Run All, sans données locales, réseau, R ou TensorFlow.
**Mode B — FULL REPRODUCTION** : procédure manuelle isolée dans
[`reproduction_guide.md`](../docs/reproduction_guide.md), jamais lancée par ce notebook.
Le holdout consommé ne doit pas servir à retuner les modèles.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# Lancer depuis la racine ou notebooks/ ; aucun chemin propre à un PC.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/phase9a_frozen_artifacts_manifest.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Ouvrir ce notebook dans le repository complet.")
sys.path.insert(0, str(ROOT / "python"))
from phase9a_review import digest, metric_replay, published, table, verify_frozen

MODE = "FAST"
assert MODE == "FAST", "FULL est une procédure isolée, pas une option Run All."
frozen_before = verify_frozen(ROOT, portable=True)
assert frozen_before.status.eq("PASS").all(), frozen_before.query("status != 'PASS'")
display(frozen_before.groupby(["status", "representation"]).size().rename("files").to_frame())

def show_figure(relative):
    path = published(relative, ROOT)
    pixels = mpimg.imread(path)
    assert pixels.size > 0 and pixels.ndim == 3
    display(Image(data=path.read_bytes(), width=950))

print("FAST : sources gelées vérifiées ; aucune lecture de data/.")

## Data
Yahoo Finance via yfinance, requête du **2018-01-01 au 2026-10-02** inclus
(`end=2026-10-03`, exclusif). Les bornes observées et le nombre de lignes
figurent dans les manifestes ci-dessous. `auto_adjust=False` conserve les champs
vendor ; Yahoo a déjà ajusté le Close historique pour les splits. Adj Close
inclut les ajustements pertinents, notamment distributions.

Dataset canonique : intersection des séances observées, `Date` + six champs
OHLC/Adj_Close/Volume par actif, 19 colonnes, sans imputation ni arrondi.
Les données brutes et préparées sont ignorées par Git ; FAST ne les charge pas.
Un téléchargement Yahoo ultérieur peut changer les valeurs historiques.
Un manifeste ne permet pas de reconstruire les octets d'un snapshot absent.

Transformations : log-prix ln(P), log-rendement 100×Δln(P), rendement simple
100×(P/P_précédent−1), variation relative du volume, volatilité historique
sur les 20 derniers rendements (écart type ddof=1 ×√252).
Les NA structurels initiaux sont conservés. Le split intervient avant leur exclusion.
Source : [`data_dictionary.md`](../docs/data_dictionary.md).

In [ ]:
# Tableau historique versionné : outputs/tables/phase1_raw_manifest.csv
display(table('phase1_raw_manifest', ROOT)[['ticker', 'rows', 'first_date', 'last_date', 'sha256']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase1_5_processed_manifest.csv
display(table('phase1_5_processed_manifest', ROOT)[['rows', 'columns', 'first_date', 'last_date', 'sha256', 'alignment_method']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase2_feature_manifest.csv
display(table('phase2_feature_manifest', ROOT)[['rows', 'columns', 'derived_variable_count', 'sha256', 'train_sha256', 'test_sha256']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase2_split_metadata.csv
display(table('phase2_split_metadata', ROOT))

In [ ]:
split = table("phase2_split_metadata", ROOT).iloc[0]
assert split.train_rows == int(np.floor(0.90 * split.total_rows))
assert split.train_rows + split.test_rows == split.total_rows
assert split.train_end < split.test_start
assert split.train_end < split.test_end
print(f"Split contrôlé : {split.train_rows} TRAIN / {split.test_rows} TEST, chronologique.")

## EDA
Les figures de niveaux/rendements couvrent la période descriptive et signalent
le holdout. Les diagnostics qui influencent le développement utilisent TRAIN.
Les corrélations contemporaines ne prouvent aucune causalité économique.
ACF/PACF : 40 retards, bandes ponctuelles ; ADF H0=racine unitaire,
KPSS H0=stationnarité. Constante+tendance pour les niveaux et constante pour
les rendements ; les avertissements/bornes KPSS sont conservés dans les sources.
Les deux tests soutiennent des niveaux non stationnaires et des rendements
stationnaires selon ces spécifications, sans garantir l'invariance des régimes.
Source : [`phase2_eda_stationarity.md`](../docs/phase2_eda_stationarity.md).

**Figure historique versionnée** : `outputs/figures/phase2/normalized_prices_base100.png`. Comparer les trajectoires à base initiale commune, sans scaler ajusté sur TEST.

In [ ]:
show_figure('outputs/figures/phase2/normalized_prices_base100.png')

**Figure historique versionnée** : `outputs/figures/phase2/log_returns.png`. Concentrations de variations ; ne pas attribuer les chocs à des causes non testées.

In [ ]:
show_figure('outputs/figures/phase2/log_returns.png')

In [ ]:
# Tableau historique versionné : outputs/tables/phase2_return_correlation_train.csv
display(table('phase2_return_correlation_train', ROOT))

**Figure historique versionnée** : `outputs/figures/phase2/nvda_adj_close_acf_pacf_train.png`. Persistance du niveau sur TRAIN.

In [ ]:
show_figure('outputs/figures/phase2/nvda_adj_close_acf_pacf_train.png')

**Figure historique versionnée** : `outputs/figures/phase2/nvda_log_return_acf_pacf_train.png`. Dynamique des différences logarithmiques ; ce ne sont pas des différences en dollars.

In [ ]:
show_figure('outputs/figures/phase2/nvda_log_return_acf_pacf_train.png')

In [ ]:
# Tableau historique versionné : outputs/tables/phase2_stationarity_summary.csv
display(table('phase2_stationarity_summary', ROOT))

In [ ]:
# Tableau historique versionné : outputs/tables/phase2_stationarity_tests.csv
display(table('phase2_stationarity_tests', ROOT)[['ticker', 'series', 'scope', 'test', 'test_statistic', 'p_value', 'decision_5pct']])

## ARIMA
La série modélisée est le log-prix NVDA. **d=1** est justifié par les diagnostics
de TRAIN. L'exploration hebdomadaire/STL ne justifie pas une SARIMA ; on ne
force pas une saisonnalité sur un calendrier de séances avec jours fériés.
La Phase 3A compare identification manuelle et automatique, IC, convergence,
Ljung–Box, normalité et ARCH. Des sensibilités d'optimiseur et diagnostics
défavorables restent documentés.

La validation glissante interne Phase 3B compare des origines identiques h5/h10,
sans ouvrir le holdout pour sélectionner. Aucun challenger ne satisfait la
règle complète R/Python : **Naive / ARIMA(0,1,0) sans drift est verrouillé**,
avec son avertissement d'autocorrélation résiduelle conservé.
ARIMA(2,1,2) avec drift demeure un challenger pédagogique préspécifié en Phase 7 ;
cela ne remplace pas le verrou Phase 3B.
Sources : [identification](../docs/phase3a_arima_identification.md),
[validation](../docs/phase3b_arima_validation.md).

In [ ]:
# Tableau historique versionné : outputs/tables/phase3_arima_diagnostic_summary.csv
display(table('phase3_arima_diagnostic_summary', ROOT)[['language', 'model', 'ljung_box_10', 'ljung_box_20', 'normality_supported', 'arch_effect_supported', 'qualification']])

**Figure historique versionnée** : `outputs/figures/phase3/nvda_arima_checkresiduals_r.png`. Diagnostics R historiques ; une faible AIC ne suffit pas à valider le modèle.

In [ ]:
show_figure('outputs/figures/phase3/nvda_arima_checkresiduals_r.png')

In [ ]:
# Tableau historique versionné : outputs/tables/phase3b_arima_validation_metrics.csv
display(table('phase3b_arima_validation_metrics', ROOT))

In [ ]:
# Tableau historique versionné : outputs/tables/phase3b_locked_arima_model.csv
display(table('phase3b_locked_arima_model', ROOT))

**Figure historique versionnée** : `outputs/figures/phase3b/phase3b_rmse_by_model_h10.png`. Validation interne TRAIN, distincte des scores finaux.

In [ ]:
show_figure('outputs/figures/phase3b/phase3b_rmse_by_model_h10.png')

## VAR / Granger / Johansen
Un **VAR(9)** constant porte sur les trois log-rendements en pourcentage.
Sa stabilité est vérifiée à partir des valeurs propres de la matrice compagnon.
Les tests Granger conditionnels examinent une information prédictive incrémentale,
avec correction Holm, et non une preuve de causalité réelle.
Johansen porte sur les log-prix en niveaux avec une convention de retards distincte :
le rang retenu est **nul**, donc **VECM non justifié**.
Les IRF/FEVD orthogonalisées sont descriptives, sensibles à l'ordre de Cholesky
NVDA, AMD, SOXX ; aucune identification causale structurelle n'est revendiquée.
Les avertissements de normalité/hétéroscédasticité restent valables.
Source : [`phase4_var_vecm.md`](../docs/phase4_var_vecm.md).

In [ ]:
# Tableau historique versionné : outputs/tables/phase4_var_stability.csv
display(table('phase4_var_stability', ROOT)[['lag', 'language', 'companion_modulus', 'stable']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase4_granger_causality_comparison.csv
display(table('phase4_granger_causality_comparison', ROOT)[['cause', 'effect', 'p_value_r', 'p_value_holm_r', 'p_value_python', 'p_value_holm_python', 'decision_agrees']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase4_vecm_decision.csv
display(table('phase4_vecm_decision', ROOT)[['cointegration_rank', 'rank_r', 'rank_python', 'vecm_justified', 'reason', 'level_lags', 'k_ar_diff']])

**Figure historique versionnée** : `outputs/figures/phase4/phase4_var_irf.png`. Réponses orthogonalisées aux innovations, dépendantes de l'ordre.

In [ ]:
show_figure('outputs/figures/phase4/phase4_var_irf.png')

**Figure historique versionnée** : `outputs/figures/phase4/phase4_var_fevd.png`. Décomposition descriptive de variance des erreurs de prévision.

In [ ]:
show_figure('outputs/figures/phase4/phase4_var_fevd.png')

## GARCH
ARCH-LM préalable sur les innovations du filtre moyen ARMA(1,1) justifie
la modélisation de variance ; ce filtre ne remplace pas le benchmark de prix.
GARCH(1,1) Normal et Student-t sont comparés dans R et Python.
Student-t améliore fortement les IC, mais la règle gelée exige aussi une
amélioration du score de queues dans les deux langues : cette condition échoue
en R. Le verrou commun reste **Normal**, sans prétendre que les queues sont gaussiennes.

La persistance α+β est élevée et inférieure à un. Les diagnostics sur z et z²
ne détectent plus d'ARCH aux retards testés ; une non-rejection ne prouve pas iid.
Les sigmas ajustés sur tout TRAIN sont rétrospectifs, pas des features temps réel.
Les prévisions de sigma à la fin de TRAIN concernent le **risque**, pas la direction
du prix, ni une volatilité cumulée sur dix séances. Aucun score de volatilité
sur holdout n'a été publié. Source : [`phase5_garch.md`](../docs/phase5_garch.md).

In [ ]:
# Tableau historique versionné : outputs/tables/phase5_arch_lm_pre_garch.csv
display(table('phase5_arch_lm_pre_garch', ROOT)[['language', 'lag', 'statistic', 'p_value']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase5_garch_model_comparison.csv
display(table('phase5_garch_model_comparison', ROOT)[['language', 'model', 'distribution', 'AIC', 'BIC', 'persistence', 'tail_quantile_rmse', 'selected']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase5_locked_garch_model.csv
display(table('phase5_locked_garch_model', ROOT)[['locked_garch_specification', 'selection_reason']])

**Figure historique versionnée** : `outputs/figures/phase5/phase5_conditional_volatility.png`. Persistance des épisodes de risque sur TRAIN, sigma quotidienne en pourcentage.

In [ ]:
show_figure('outputs/figures/phase5/phase5_conditional_volatility.png')

**Figure historique versionnée** : `outputs/figures/phase5/phase5_squared_standardized_residual_acf.png`. Contrôle de dépendance de variance après ajustement.

In [ ]:
show_figure('outputs/figures/phase5/phase5_squared_standardized_residual_acf.png')

In [ ]:
# Tableau historique versionné : outputs/tables/phase5_volatility_forecast_train_end.csv
display(table('phase5_volatility_forecast_train_end', ROOT)[['language', 'model', 'horizon', 'forecast_variance', 'forecast_sigma', 'locked']])

In [ ]:
volatility = table("phase5_volatility_forecast_train_end", ROOT)
np.testing.assert_allclose(volatility.forecast_sigma ** 2, volatility.forecast_variance,
                           rtol=1e-12, atol=1e-12)
print("Contrôle arithmétique : sigma² = variance des prévisions archivées.")

## LSTM / GRU
Cinq features : log-prix NVDA, log-rendement NVDA, volatilité historique
NVDA sur 20 séances, log-rendements AMD et SOXX. Exclusion des seuls 20 NA
initiaux ; les sigmas GARCH ajustés sur tout TRAIN ne sont pas utilisés.
Sliding windows : entrées [t−L+1,…,t] et dix cibles [t+1,…,t+10].
StandardScaler X/y appris sur le préfixe TRAIN de chaque fold, jamais sur TEST.
Trois folds temporels en expansion ; early stopping sur une queue interne à TRAIN,
avec purge des fenêtres traversant la frontière interne.

Input → LSTM ou GRU → Dropout → Dense(10), sorties directes log-prix,
Adam et MSE standardisée. La grille gelée A/B/C conduit à C pour les deux
réseaux ; le protocole et les verrous affichés ci-dessous donnent les paramètres exacts.
Les réseaux perdent contre Naive en validation interne ; leur présence obligatoire
n'implique pas une supériorité. Le coût CPU et les courbes de loss sont conservés.
Les loss affichées correspondent à la validation interne d'early stopping,
pas aux folds externes ni au holdout. Source : [`phase6_lstm_gru.md`](../docs/phase6_lstm_gru.md).

In [ ]:
neural_protocol = json.loads(published("docs/phase6_protocol.json", ROOT).read_text(encoding="utf8"))
display(neural_protocol)

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_temporal_folds.csv
display(table('phase6_temporal_folds', ROOT))

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_scaler_audit.csv
display(table('phase6_scaler_audit', ROOT))

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_locked_lstm.csv
display(table('phase6_locked_lstm', ROOT)[['configuration', 'lookback', 'units', 'dropout', 'trainable_parameters', 'median_best_epoch', 'average_RMSE_ratio']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_locked_gru.csv
display(table('phase6_locked_gru', ROOT)[['configuration', 'lookback', 'units', 'dropout', 'trainable_parameters', 'median_best_epoch', 'average_RMSE_ratio']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_vs_naive.csv
display(table('phase6_vs_naive', ROOT))

In [ ]:
# Tableau historique versionné : outputs/tables/phase6_full_train_refits.csv
display(table('phase6_full_train_refits', ROOT)[['model', 'refit_epochs', 'training_seconds', 'status']])

**Figure historique versionnée** : `outputs/figures/phase6/phase6_lstm_loss_curve.png`. Configuration verrouillée C, fold 3 ; epoch restauré documenté dans les CSV.

In [ ]:
show_figure('outputs/figures/phase6/phase6_lstm_loss_curve.png')

**Figure historique versionnée** : `outputs/figures/phase6/phase6_gru_loss_curve.png`. Même dispositif temporel pour GRU.

In [ ]:
show_figure('outputs/figures/phase6/phase6_gru_loss_curve.png')

In [ ]:
# Nouvelle visualisation de séries de loss publiées, aucun entraînement.
history = table("phase6_loss_history", ROOT)
selected_history = history[(history.configuration == "C") & (history.fold == 3)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, model in zip(axes, ["LSTM", "GRU"]):
    data = selected_history[selected_history.model == model].sort_values("epoch")
    ax.plot(data.epoch, data.loss, label="Optimisation TRAIN")
    ax.plot(data.epoch, data.val_loss, label="Queue interne TRAIN")
    ax.set(title=f"{model} C / Fold 3 — CSV historique", xlabel="Epoch", ylabel="MSE standardisée")
    ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## Final Evaluation
Protocole commité avant ouverture finale : modèles/scalers figés, origines
communes h5/h10, préfixe observé jusqu'à chaque origine. ARIMA met à jour son
état filtré avec `refit=False` ; VAR reconstruit les log-prix à partir de
rendements prévus divisés par 100 ; les réseaux fournissent des sorties directes.
Les paramètres ne sont pas réestimés au cours du holdout.

**FAST ne charge ni TRAIN ni TEST et ne produit aucune nouvelle prévision.**
Le panneau ci-dessous vérifie les dates, cibles et MSE/RMSE/MAE/MAPE par
arithmétique depuis `phase7_final_predictions.csv` gelé, puis compare aux CSV
officiels. MSE en USD² ajustés, RMSE/MAE en USD ajustés, MAPE en pourcentage.
Il s'agit d'un replay des sorties, pas d'une nouvelle évaluation historique.
Source : [`phase7_final_evaluation.md`](../docs/phase7_final_evaluation.md).

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_holdout_manifest.csv
display(table('phase7_holdout_manifest', ROOT)[['train_rows', 'test_rows', 'common_origins', 'train_last_date', 'test_first_date', 'test_last_date']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_prediction_freeze.csv
display(table('phase7_prediction_freeze', ROOT)[['rows', 'predictions_sha256', 'frozen_utc']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_final_metrics.csv
display(table('phase7_final_metrics', ROOT)[['model', 'horizon', 'n_forecasts', 'MSE', 'RMSE', 'MAE', 'MAPE']])

In [ ]:
replayed_metrics = metric_replay(ROOT)
display(replayed_metrics)
print("PASS : mêmes origines/cibles ; quatre métriques conformes aux CSV gelés.")

Diebold–Mariano publié : pertes quadratiques, différences A−B, HAC Bartlett
de retard h−1, correction HLN, référence t(n−1). Holm traite quatre comparaisons
vs Naive séparément par horizon ; GRU vs LSTM est une comparaison brute distincte.
Les origines se chevauchent et ne sont pas indépendantes. **p>0.05 ne prouve pas
l'équivalence**. FAST affiche la table existante et ne recalcule pas DM.

ARIMA a une légère avance RMSE à h10, mais échoue à la règle complète gelée
de gains, MAE et soutien DM. Aucun challenger ne remplit toutes les conditions :
**Naive reste recommandé**. Les résultats défavorables LSTM/GRU sont conservés.
GARCH concerne une autre cible et est exclu du classement prix.

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_diebold_mariano.csv
display(table('phase7_diebold_mariano', ROOT)[['model_a', 'model_b', 'horizon', 'mean_loss_diff', 'dm_statistic', 'p_value', 'p_value_holm_if_applicable', 'decision_5pct']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_final_model_recommendation.csv
display(table('phase7_final_model_recommendation', ROOT)[['final_recommended_price_model', 'rule', 'reason']])

In [ ]:
# Tableau historique versionné : outputs/tables/phase7_model_tradeoffs.csv
display(table('phase7_model_tradeoffs', ROOT))

**Figure historique versionnée** : `outputs/figures/phase7/phase7_rmse_comparison_h5.png`. Classement observé ; aucune nouvelle règle de sélection.

In [ ]:
show_figure('outputs/figures/phase7/phase7_rmse_comparison_h5.png')

**Figure historique versionnée** : `outputs/figures/phase7/phase7_rmse_comparison_h10.png`. Distinguer meilleur RMSE isolé et recommandation globale.

In [ ]:
show_figure('outputs/figures/phase7/phase7_rmse_comparison_h10.png')

**Figure historique versionnée** : `outputs/figures/phase7/phase7_actual_vs_forecasts_h10.png`. Panneau de prévisions archivées, mêmes dates cibles et même benchmark.

In [ ]:
show_figure('outputs/figures/phase7/phase7_actual_vs_forecasts_h10.png')

## LLM
Prompt officiel : [`phase8_official_llm_prompt.md`](../llm/prompts/phase8_official_llm_prompt.md),
avec facts sourcés et disclaimer. Quatre demandes : A hypothèses de modèles,
B vulgarisation, C interprétation financière simulée, D revue critique.
Réponse officielle : [`phase8_official_llm_response.md`](../llm/responses/phase8_official_llm_response.md),
préservée intégralement. Le modèle exact et la date de génération ne sont pas
enregistrés : ils restent inconnus, le commit n'est pas une date de génération.

La réponse synthétise la stationnarité, les dépendances, le risque GARCH,
les résultats neuronaux défavorables et la recommandation Naive. Elle a reçu
ces résultats après les analyses : cette restitution n'est pas une capacité de
prévision prospective. Les baselines sont des interprétations de développement
revues par l'équipe ; leur indépendance de l'IA n'est pas démontrée.

La comparaison complète garde les accords, différences de précision et limites.
L'audit des claims ne détecte pas d'hallucination numérique parmi les valeurs
contrôlées. Des citations locales imprécises et un suffixe éditorial restent
documentés ; les catégories de claims ne mesurent pas la fiabilité générale des LLM.
La simulation financière ne fournit ni allocation, ni buy/sell, ni rendement
garanti ; volatilité n'est pas direction. Source : [`phase8_llm_analysis.md`](../docs/phase8_llm_analysis.md).

In [ ]:
prompt_path = published("llm/prompts/phase8_official_llm_prompt.md", ROOT)
response_path = published("llm/responses/phase8_official_llm_response.md", ROOT)
comparison_path = published("llm/human_analysis/phase8_human_vs_llm_comparison.md", ROOT)
prompt = prompt_path.read_text(encoding="utf8")
response = response_path.read_text(encoding="utf8")
comparison = comparison_path.read_text(encoding="utf8")
display(pd.DataFrame([{"artifact": str(p.relative_to(ROOT)), "sha256": digest(p)}
                      for p in [prompt_path, response_path, comparison_path]]))
# Prévisualisation contrôlée ; lire les fichiers liés pour les textes complets.
display(Markdown(prompt[:1800] + "\n\n… [Prompt complet dans le fichier source]"))
display(Markdown(comparison[:1800] + "\n\n… [Comparaison complète dans le fichier source]"))
audit = pd.read_csv(published("llm/human_analysis/phase8_hallucination_audit.csv", ROOT))
display(audit)
display(audit.groupby(["issue_type", "severity"]).size().rename("claims").to_frame())

## Conclusion
Les diagnostics de TRAIN justifient les transformations, le VAR et le GARCH,
et motivent des approches complémentaires. Les réseaux essayés ne battent pas
le benchmark. La recommandation finale reste Naive sous la règle préspécifiée,
sans affirmer sa supériorité universelle. La qualité d'une interprétation LLM
se juge sur les preuves, la précision et la prudence, avec provenance incomplète.

Limites : un seul régime de holdout, erreurs chevauchantes, queues épaisses,
incertitude des paramètres, nonnormalité/hétéroscédasticité, CPU dépendant de
la machine, un seul seed neural, prix ajustés et absence de coûts/backtest.
Yahoo peut réviser les snapshots ; TensorFlow ne garantit pas une identité
bit-for-bit entre machines malgré seeds et opérations déterministes demandées.

Perspectives pour une étude future indépendante : nouveaux régimes, données
archivées distribuables, environnement verrouillé, validation multi-seed et
vrais tests économiques. **Aucune de ces expériences n'est menée en Phase 9A.**
Phase 9B reste la rédaction du rapport de 15–25 pages à partir des résultats existants.

Pour l'équipe : [Windows/RStudio](../docs/team_setup_windows.md),
[reproduction FAST/FULL](../docs/reproduction_guide.md),
[checklist](../docs/final_requirements_checklist.md),
[figures pour le rapport](../docs/final_figure_selection.md).

In [ ]:
frozen_after = verify_frozen(ROOT, portable=True)
assert frozen_after.status.eq("PASS").all()
pd.testing.assert_frame_equal(frozen_before, frozen_after)
assert table("phase7_final_model_recommendation", ROOT).iloc[0].final_recommended_price_model == "Naive"
print(f"FAST terminé : {len(frozen_after)} artefacts inchangés ; modèle officiel Naive conservé.")